In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

In [27]:
import os

data_path = "/Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun"

files = [f for f in os.listdir(data_path) if f.endswith(".csv")]

tables = {}
for f in files:
    name = f.replace(".csv", "")
    tables[name] = pd.read_csv(os.path.join(data_path, f))

tables.keys()

dict_keys(['circuits', 'weather_data', 'f1_master_enriched', 'status', 'lap_times', 'sprint_results', 'drivers', 'races', 'constructors', 'constructor_standings', 'qualifying', 'driver_standings', 'constructor_results', 'pit_stops', 'seasons', 'results'])

In [8]:
#pip install meteostat==1.6.5

  Attempting uninstall: meteostat
    Found existing installation: meteostat 2.1.4
    Uninstalling meteostat-2.1.4:
      Successfully uninstalled meteostat-2.1.4
Note: you may need to restart the kernel to use updated packages.


In [28]:
results = tables["results"].copy()
races = tables["races"].copy()
drivers = tables["drivers"].copy()
constructors = tables["constructors"].copy()
circuits = tables["circuits"].copy()
status = tables["status"].copy()

# Clean duplicate columns
for df in [races, drivers, constructors, circuits]:
    if "url" in df.columns:
        df.drop(columns=["url"], inplace=True)

races["date"] = pd.to_datetime(races["date"])

master = (
    results
    .merge(races, on="raceId", how="left")
    .merge(drivers, on="driverId", how="left")
    .merge(constructors, on="constructorId", how="left")
    .merge(circuits, on="circuitId", how="left")
    .merge(status, on="statusId", how="left")
)

master.shape

(26759, 52)

In [1]:
# pip install meteostat

Note: you may need to restart the kernel to use updated packages.


In [5]:
#pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl (20.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.0
    Uninstalling numpy-2.3.0:
      Successfully uninstalled numpy-2.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [29]:
import requests
import pandas as pd

def fetch_open_meteo_daily(lat, lon, date_str):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_str,
        "end_date": date_str,
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "wind_speed_10m_max",
            "relative_humidity_2m_mean"
        ],
        "timezone": "GMT",
        "models": "era5"
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    daily = data.get("daily", {})
    if not daily or len(daily.get("time", [])) == 0:
        return None

    return {
        "temp_avg": daily.get("temperature_2m_mean", [None])[0],
        "temp_max": daily.get("temperature_2m_max", [None])[0],
        "temp_min": daily.get("temperature_2m_min", [None])[0],
        "precipitation": daily.get("precipitation_sum", [None])[0],
        "wind_speed": daily.get("wind_speed_10m_max", [None])[0],
        "humidity_avg": daily.get("relative_humidity_2m_mean", [None])[0],
    }

In [30]:
race_weather = (
    master[["raceId", "date", "lat", "lng", "name_x", "year"]]
    .drop_duplicates()
    .dropna(subset=["date", "lat", "lng"])
    .copy()
)

weather_rows = []
failures = []

for _, row in race_weather.iterrows():
    try:
        date_str = pd.Timestamp(row["date"]).strftime("%Y-%m-%d")
        weather = fetch_open_meteo_daily(row["lat"], row["lng"], date_str)

        if weather is None:
            failures.append({
                "raceId": row["raceId"],
                "race_name": row["name_x"],
                "year": row["year"],
                "reason": "no_daily_data"
            })
            continue

        weather_rows.append({
            "raceId": row["raceId"],
            "race_name": row["name_x"],
            "year": row["year"],
            **weather
        })

    except Exception as e:
        failures.append({
            "raceId": row["raceId"],
            "race_name": row["name_x"],
            "year": row["year"],
            "reason": str(e)
        })

weather_df = pd.DataFrame(weather_rows)
failures_df = pd.DataFrame(failures)

print(weather_df.shape)
print(failures_df.shape)
weather_df.head()

(1123, 9)
(2, 4)


,raceId,race_name,year,temp_avg,temp_max,temp_min,precipitation,wind_speed,humidity_avg
0,18,Australian Grand Prix,2008,27.9,38.4,18.1,0.0,23.1,41
1,19,Malaysian Grand Prix,2008,26.7,30.5,23.9,9.6,8.5,86
2,20,Bahrain Grand Prix,2008,23.1,23.4,22.7,0.0,24.3,66
3,21,Spanish Grand Prix,2008,15.6,19.2,11.3,0.0,12.8,73
4,22,Turkish Grand Prix,2008,13.6,18.3,8.2,0.0,16.9,70


In [35]:
master = master.merge(
    weather_df.drop(columns=["race_name", "year"]),
    on="raceId",
    how="left"
)

In [36]:
master[["temp_avg", "temp_max", "temp_min", "precipitation", "wind_speed", "humidity_avg"]].isna().mean()

temp_avg         0.001831
temp_max         0.001831
temp_min         0.001831
precipitation    0.001831
wind_speed       0.001831
humidity_avg     0.001831
dtype: float64

In [37]:
master["month"] = pd.to_datetime(master["date"]).dt.month
master["temp_range"] = master["temp_max"] - master["temp_min"]
master["is_wet_race"] = (master["precipitation"] > 0).astype(int)
master["abs_lat"] = master["lat"].abs()
master["high_altitude_track"] = (master["alt"] >= 500).astype(int)

In [38]:
data_path = "/Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun"


import os

master.to_csv(os.path.join(data_path, "f1_master_enriched.csv"), index=False)
weather_df.to_csv(os.path.join(data_path, "weather_data.csv"), index=False)

In [18]:
import os
os.path.exists(data_path)

True

In [20]:
os.makedirs("data_processed", exist_ok=True)

In [39]:
master.to_csv("data_processed/f1_master_enriched.csv", index=False)
weather_df.to_csv("data_processed/weather_data.csv", index=False)

In [22]:
mkdir -p data_processed

In [23]:
os.getcwd()

'/Users/paigeblackstone/Documents/GitHub/f1-worldchamp'

In [24]:
os.listdir()

['F1_models_iteration1.ipynb',
 'data_processed',
 'requirements.txt',
 'gameplan.txt',
 'README.md',
 'initial_script.ipynb',
 '.gitignore',
 'weather_data.ipynb',
 '.git',
 'src']

In [25]:
os.makedirs("data_processed", exist_ok=True)